# Day 08 - Date Timestamp and String Functions

**Topic:** Date Timestamp and String Functions  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึก parse และ standardize date/time/string fields ด้วย built-in PySpark functions เพื่อเตรียมข้อมูลให้พร้อมใช้ใน Lakehouse pipeline

วันนี้เราจะฝึกจัดการข้อมูลประเภท date, timestamp และ string ที่มักเจอใน raw source เช่น date หลาย format, timestamp จากหลายระบบ, email/phone/status ที่เขียนไม่สม่ำเสมอ และ source reference ที่ต้อง split ออกเป็น column ใช้งานจริง


## Goal of the day

1. แยกให้ออกว่า **Date**, **Timestamp** และ **String** field ควรถูก parse หรือ standardize อย่างไร
2. ใช้ `to_date`, `to_timestamp`, `date_format`, `datediff`, `current_timestamp` ได้อย่างถูกจุด
3. ใช้ string functions เช่น `trim`, `lower`, `upper`, `regexp_replace`, `split`, `concat` เพื่อ clean raw text fields
4. ตรวจ records ที่ parse date/timestamp ไม่ได้ โดยไม่ drop เงียบ ๆ
5. สร้าง standardized DataFrame ที่พร้อมนำไปใช้ต่อใน Lakehouse table หรือ downstream transformation


## Why it matters in production

ใน production pipeline ข้อมูล date/time/string มักเป็นจุดที่ทำให้ data quality เพี้ยนได้ง่าย เพราะ:

- source system แต่ละตัวอาจส่ง date มาคนละ format เช่น `yyyy-MM-dd`, `dd/MM/yyyy`, `yyyy/MM/dd`
- timestamp อาจเป็น event time จาก source หรือ processing time ของ pipeline ซึ่งใช้คนละความหมาย
- string fields เช่น email, status, phone, customer code มักมีช่องว่าง ตัวพิมพ์เล็ก/ใหญ่ หรือ special characters ปนมา
- ถ้า parse date ผิดหรือได้ `null` โดยไม่รู้ตัว อาจกระทบ partition, watermark, SLA, report period และ business logic
- ถ้าไม่ normalize string ก่อน join/filter อาจทำให้ records match ไม่ครบ

Production takeaway:

> Raw string ไม่ควรถูกเชื่อทันที ต้อง parse, standardize และตรวจ invalid records ให้มองเห็นได้ก่อนใช้ต่อ


## Key concepts

| Concept | Meaning |
|---|---|
| `to_date` | แปลง string เป็น DateType ตาม format ที่กำหนด |
| `to_timestamp` | แปลง string เป็น TimestampType ตาม format ที่กำหนด |
| `date_format` | แปลง date/timestamp เป็น string format สำหรับ reporting หรือ partition helper |
| `datediff` | คำนวณจำนวนวันระหว่าง date สองค่า |
| `current_timestamp` | เวลาที่ Spark job กำลังประมวลผล ใช้เป็น processing timestamp ได้ |
| `trim` | ตัดช่องว่างหัวท้าย string |
| `lower` / `upper` | normalize ตัวพิมพ์เล็ก/ใหญ่ให้สม่ำเสมอ |
| `regexp_replace` | replace string ด้วย regular expression เช่น เก็บเฉพาะตัวเลขในเบอร์โทร |
| `split` | แยก string ออกเป็น array ด้วย delimiter |
| `concat` / `concat_ws` | รวม string หลายส่วนเป็นค่าใหม่ เช่น standardized code |
| Event time | เวลาที่ event เกิดขึ้นจริงใน source system |
| Processing time | เวลาที่ pipeline ประมวลผล record นั้น |


## Concept explanation

### Date vs Timestamp vs String

ใน raw data หลายครั้ง date/time จะเริ่มต้นเป็น `StringType` ก่อน เช่น:

```text
2026-05-01
01/05/2026
2026/05/01
2026-05-01 09:15:30
```

ถ้าเก็บเป็น string ต่อไป จะมีปัญหาเวลา filter, sort, partition หรือคำนวณช่วงเวลา เพราะ Spark จะมองเป็น text ไม่ใช่ date/time จริง

แนวคิดหลักคือ:

> Raw string → Parse เป็น Date/Timestamp → Validate parse result → Derive columns ที่ต้องใช้ต่อ

### Event time vs Processing time

สองคำนี้สำคัญมากใน pipeline จริง:

- **Event time** คือเวลาที่ transaction หรือ event เกิดขึ้นจริง เช่น `transaction_date`, `event_timestamp`
- **Processing time** คือเวลาที่ Spark job run หรือ ingest ข้อมูล เช่น `processing_timestamp`

อย่าใช้ `current_timestamp()` แทน event time โดยไม่ตั้งใจ เพราะจะทำให้ business timeline เพี้ยน

### String standardization

String fields ที่ใช้ join/filter/group ควร standardize ก่อน เช่น:

- `trim()` เพื่อตัดช่องว่างหัวท้าย
- `lower()` สำหรับ email หรือ status
- `upper()` สำหรับ code ที่ควรเป็น uppercase
- `regexp_replace()` สำหรับลบ special characters หรือ normalize phone/account number
- `split()` เพื่อแตก composite string เช่น `web|checkout|BKK`
- `concat()` เพื่อสร้าง standardized key/code

### Invalid parse ต้องมองเห็นได้

ถ้า date string parse ไม่ได้ อย่า drop record ทิ้งทันที ควรสร้าง flag เช่น `date_parse_status` เพื่อให้รู้ว่า record ไหนต้อง review ต่อ

ตัวอย่าง mindset:

> Parse failure คือ data quality signal ไม่ใช่แค่ error เล็ก ๆ ที่ควรถูกซ่อน


## Mermaid diagram: Date and String Standardization Flow

```mermaid
flowchart LR
    A[Raw source strings] --> B[Trim and normalize text]
    B --> C[Parse date and timestamp]
    C --> D{Parse success?}
    D -- Yes --> E[Derive year month day and clean columns]
    D -- No --> F[Flag parse issue for review]
    E --> G[Standardized DataFrame]
    F --> G
```

Key idea:

> การ clean date/time/string ควรเก็บทั้ง raw value และ standardized value อย่างน้อยในช่วงเรียนรู้หรือ debug เพื่อให้ trace กลับได้


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 3, Finished, Available, Finished, False)

## Create mock data

Mock data วันนี้ตั้งใจให้มี raw fields ที่ไม่เรียบร้อย เช่น:

- date หลาย format
- timestamp หลาย format
- email/status ที่มีช่องว่างและตัวพิมพ์ปนกัน
- phone number ที่มี `-`, space, `+`, `()`
- source reference ที่รวมหลายค่าไว้ใน string เดียว


In [2]:
transactions_raw_data = [
    (
        "T001", 101, "  Alice Wong ", "ALICE.WONG@EXAMPLE.COM ", "+66 81-234-5678",
        " success ", "bkk-001", "2026-05-01", "2026-05-01 09:15:30", "web|checkout|BKK", 1200.50
    ),
    (
        "T002", 102, "bob chan", " Bob.Chan@example.com", "081 999 0000",
        "FAILED", "bkk-002", "01/05/2026", "01/05/2026 10:05:00", "mobile|payment|BKK", 0.00
    ),
    (
        "T003", 103, " CHARLIE TAN", "CHARLIE.TAN@EXAMPLE.COM", "(+66)82 555 8899",
        "Success", "cnx-003", "2026/05/02", "2026/05/02 14:20:00", "partner|booking|CNX", 3400.00
    ),
    (
        "T004", 104, "dara lee ", " dara.lee@example.com ", "083-777-1111",
        " pending", "hkt-004", "2026-05-03", "2026-05-03T09:00:00", "web|payment|HKT", 780.25
    ),
    (
        "T005", 105, "Eve Kim", "EVE.KIM@EXAMPLE.COM", "084 123 4567",
        "SUCCESS", "bkk-005", "not_available", "not_available", "mobile|checkout|BKK", 560.00
    ),
    (
        "T006", 106, "  frank park", "FRANK.PARK@example.com ", "66851234567",
        "success", "ryg-006", "2026-05-04", "2026-05-04 18:45:10", "branch|counter|RYG", 2200.00
    ),
    (
        "T007", 107, "Grace Lin", " grace.lin@example.com", "085-000-1111",
        "failed ", "cnx-007", "", "", "web|checkout|CNX", 150.00
    ),
]

transaction_raw_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name_raw", T.StringType(), True),
    T.StructField("email_raw", T.StringType(), True),
    T.StructField("phone_raw", T.StringType(), True),
    T.StructField("status_raw", T.StringType(), True),
    T.StructField("customer_code_raw", T.StringType(), True),
    T.StructField("transaction_date_raw", T.StringType(), True),
    T.StructField("event_timestamp_raw", T.StringType(), True),
    T.StructField("source_reference", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
])

df_transactions_raw = spark.createDataFrame(transactions_raw_data, transaction_raw_schema)

df_transactions_raw.show(truncate=False)
df_transactions_raw.printSchema()
print("Raw row count:", df_transactions_raw.count())

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 4, Finished, Available, Finished, False)

+--------------+-----------+-----------------+-----------------------+----------------+----------+-----------------+--------------------+-------------------+-------------------+------+
|transaction_id|customer_id|customer_name_raw|email_raw              |phone_raw       |status_raw|customer_code_raw|transaction_date_raw|event_timestamp_raw|source_reference   |amount|
+--------------+-----------+-----------------+-----------------------+----------------+----------+-----------------+--------------------+-------------------+-------------------+------+
|T001          |101        |  Alice Wong     |ALICE.WONG@EXAMPLE.COM |+66 81-234-5678 | success  |bkk-001          |2026-05-01          |2026-05-01 09:15:30|web|checkout|BKK   |1200.5|
|T002          |102        |bob chan         | Bob.Chan@example.com  |081 999 0000    |FAILED    |bkk-002          |01/05/2026          |01/05/2026 10:05:00|mobile|payment|BKK |0.0   |
|T003          |103        | CHARLIE TAN     |CHARLIE.TAN@EXAMPLE.COM|(+66)

## PySpark code examples

ใน section นี้จะเริ่มจากการ clean string fields ก่อน แล้วค่อย parse date/timestamp และสร้าง standardized DataFrame ที่พร้อมใช้งานต่อ


### Example 1: Normalize basic string fields

ตัวอย่างนี้ใช้ built-in functions เพื่อ clean string fields ที่ใช้บ่อย:

- `trim` ตัดช่องว่างหัวท้าย
- `lower` normalize email/status
- `upper` normalize code
- `regexp_replace` เก็บเฉพาะตัวเลขจากเบอร์โทร
- `concat` สร้าง standardized customer key


In [ ]:
df_string_cleaned = (
    df_transactions_raw
    .withColumn("customer_name_clean", F.initcap(F.trim(F.col("customer_name_raw"))))
    .withColumn("email_clean", F.lower(F.trim(F.col("email_raw"))))
    .withColumn("status_clean", F.lower(F.trim(F.col("status_raw"))))
    .withColumn("customer_code_clean", F.upper(F.trim(F.col("customer_code_raw"))))
    .withColumn("phone_digits", F.regexp_replace(F.col("phone_raw"), "[^0-9]", ""))
    .withColumn(
        "phone_normalized",
        F.when(F.col("phone_digits").rlike("^66"), F.concat(F.lit("+"), F.col("phone_digits")))
         .when(F.col("phone_digits").rlike("^0"), F.concat(F.lit("+66"), F.substring(F.col("phone_digits"), 2, 20)))
         .otherwise(F.col("phone_digits"))
    )
    .withColumn("customer_key", F.concat(F.lit("CUST-"), F.lpad(F.col("customer_id").cast("string"), 4, "0")))
)

# Tips:
# - [^0-9] means any character that is not a digit from 0 to 9
#   Note: ^ inside  [ ]  = not

# - rlike("^66") means the value must begin with "66"
#   Note: ^ outside [ ]  = starts with

# - initcap(col): converts each word in the column to title case

# - substring(col, pos, len): extracts part of a string from the start position with the given length
#   Note: Spark substring position starts from 1, not 0

# - lpad(col, len, pad): makes a string reach the target length by adding padding characters on the left
#   Example: lpad("123", 5, "0") -> "00123"
#   Useful for standardizing IDs or codes to the same fixed length

df_string_cleaned.select(
    "transaction_id",
    "customer_name_raw",
    "customer_name_clean",
    "email_clean",
    "status_clean",
    "customer_code_clean",
    "phone_raw",
    "phone_normalized",
    "customer_key"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 5, Finished, Available, Finished, False)

+--------------+-----------------+-------------------+-----------------------+------------+-------------------+----------------+----------------+------------+
|transaction_id|customer_name_raw|customer_name_clean|email_clean            |status_clean|customer_code_clean|phone_raw       |phone_normalized|customer_key|
+--------------+-----------------+-------------------+-----------------------+------------+-------------------+----------------+----------------+------------+
|T001          |  Alice Wong     |Alice Wong         |alice.wong@example.com |success     |BKK-001            |+66 81-234-5678 |+66812345678    |CUST-0101   |
|T002          |bob chan         |Bob Chan           |bob.chan@example.com   |failed      |BKK-002            |081 999 0000    |+66819990000    |CUST-0102   |
|T003          | CHARLIE TAN     |Charlie Tan        |charlie.tan@example.com|success     |CNX-003            |(+66)82 555 8899|+66825558899    |CUST-0103   |
|T004          |dara lee         |Dara Lee    

### Example 2: Split composite source reference

บาง source ส่ง field ที่รวมหลายความหมายไว้ใน string เดียว เช่น `web|checkout|BKK`

เราสามารถใช้ `split()` เพื่อแตกออกเป็น array และดึงแต่ละตำแหน่งออกมาเป็น column ได้


In [4]:
source_parts = F.split(F.col("source_reference"), "\\|")

df_source_parsed = (
    df_string_cleaned
    .withColumn("source_system", source_parts.getItem(0))
    .withColumn("source_event", source_parts.getItem(1))
    .withColumn("source_region", source_parts.getItem(2))
    .withColumn("source_summary", F.concat_ws(" - ", "source_system", "source_event", "source_region"))
)

df_source_parsed.select(
    "transaction_id",
    "source_reference",
    "source_system",
    "source_event",
    "source_region",
    "source_summary"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 6, Finished, Available, Finished, False)

+--------------+-------------------+-------------+------------+-------------+-----------------------+
|transaction_id|source_reference   |source_system|source_event|source_region|source_summary         |
+--------------+-------------------+-------------+------------+-------------+-----------------------+
|T001          |web|checkout|BKK   |web          |checkout    |BKK          |web - checkout - BKK   |
|T002          |mobile|payment|BKK |mobile       |payment     |BKK          |mobile - payment - BKK |
|T003          |partner|booking|CNX|partner      |booking     |CNX          |partner - booking - CNX|
|T004          |web|payment|HKT    |web          |payment     |HKT          |web - payment - HKT    |
|T005          |mobile|checkout|BKK|mobile       |checkout    |BKK          |mobile - checkout - BKK|
|T006          |branch|counter|RYG |branch       |counter     |RYG          |branch - counter - RYG |
|T007          |web|checkout|CNX   |web          |checkout    |CNX          |web -

### Example 3: Parse date strings with multiple formats

ใน production จริง ไม่ควร assume ว่า source ส่ง date format เดียวเสมอ

ตัวอย่างนี้ใช้ `coalesce()` เพื่อพยายาม parse หลาย format ตามลำดับ:

1. `yyyy-MM-dd`
2. `dd/MM/yyyy`
3. `yyyy/MM/dd`

ถ้า parse ไม่ได้ทั้งหมด ผลลัพธ์จะเป็น `null` และเราจะสร้าง `date_parse_status` ไว้ตรวจต่อ


In [5]:
df_date_parsed = (
    df_source_parsed
    .withColumn(
        "transaction_date",
        F.coalesce(
            F.to_date(F.col("transaction_date_raw"), "yyyy-MM-dd"),
            F.to_date(F.col("transaction_date_raw"), "dd/MM/yyyy"),
            F.to_date(F.col("transaction_date_raw"), "yyyy/MM/dd")
        )
    )
    .withColumn(
        "date_parse_status",
        F.when(
            (F.col("transaction_date").isNull()) & (F.length(F.trim(F.col("transaction_date_raw"))) > 0),
            F.lit("unparsed")
        ).when(
            F.length(F.trim(F.col("transaction_date_raw"))) == 0,
            F.lit("missing")
        ).otherwise(F.lit("parsed"))
    )
)

# Tip:
# - coalesce checks each value in order and returns the first non-null value

df_date_parsed.select(
    "transaction_id",
    "transaction_date_raw",
    "transaction_date",
    "date_parse_status"
).show(truncate=False)

df_date_parsed.printSchema()


StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 7, Finished, Available, Finished, False)

+--------------+--------------------+----------------+-----------------+
|transaction_id|transaction_date_raw|transaction_date|date_parse_status|
+--------------+--------------------+----------------+-----------------+
|T001          |2026-05-01          |2026-05-01      |parsed           |
|T002          |01/05/2026          |2026-05-01      |parsed           |
|T003          |2026/05/02          |2026-05-02      |parsed           |
|T004          |2026-05-03          |2026-05-03      |parsed           |
|T005          |not_available       |NULL            |unparsed         |
|T006          |2026-05-04          |2026-05-04      |parsed           |
|T007          |                    |NULL            |missing          |
+--------------+--------------------+----------------+-----------------+

root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- customer_name_raw: string (nullable = true)
 |-- email_raw: string (nullable = true)
 |-- phon

### Example 4: Parse timestamp strings with multiple formats

Timestamp มักละเอียดกว่า date เพราะมีทั้งวันที่และเวลา

ตัวอย่างนี้ parse timestamp หลาย format และสร้าง `timestamp_parse_status` เพื่อให้เห็น records ที่ parse ไม่ได้


In [6]:
df_timestamp_parsed = (
    df_date_parsed
    .withColumn(
        "event_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "dd/MM/yyyy HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy/MM/dd HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy-MM-dd'T'HH:mm:ss")
        )
    )
    .withColumn(
        "timestamp_parse_status",
        F.when(
            (F.col("event_timestamp").isNull()) & (F.length(F.trim(F.col("event_timestamp_raw"))) > 0),
            F.lit("unparsed")
        ).when(
            F.length(F.trim(F.col("event_timestamp_raw"))) == 0,
            F.lit("missing")
        ).otherwise(F.lit("parsed"))
    )
)

df_timestamp_parsed.select(
    "transaction_id",
    "event_timestamp_raw",
    "event_timestamp",
    "timestamp_parse_status"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 8, Finished, Available, Finished, False)

+--------------+-------------------+-------------------+----------------------+
|transaction_id|event_timestamp_raw|event_timestamp    |timestamp_parse_status|
+--------------+-------------------+-------------------+----------------------+
|T001          |2026-05-01 09:15:30|2026-05-01 09:15:30|parsed                |
|T002          |01/05/2026 10:05:00|2026-05-01 10:05:00|parsed                |
|T003          |2026/05/02 14:20:00|2026-05-02 14:20:00|parsed                |
|T004          |2026-05-03T09:00:00|2026-05-03 09:00:00|parsed                |
|T005          |not_available      |NULL               |unparsed              |
|T006          |2026-05-04 18:45:10|2026-05-04 18:45:10|parsed                |
|T007          |                   |NULL               |missing               |
+--------------+-------------------+-------------------+----------------------+



### Example 5: Derive date columns for reporting and partition helper

หลัง parse date สำเร็จ เราสามารถ derive fields เพิ่มได้ เช่น:

- `transaction_year`
- `transaction_month`
- `transaction_day`
- `transaction_month_key`
- `days_since_transaction`

ในตัวอย่างนี้ใช้ `analysis_date` เป็นค่าคงที่เพื่อให้ผลลัพธ์ deterministic ตอนเรียนรู้ ส่วน `current_timestamp()` ใช้เป็น processing timestamp


In [7]:
analysis_date = F.to_date(F.lit("2026-05-10"), "yyyy-MM-dd")

df_date_features = (
    df_timestamp_parsed
    .withColumn("transaction_year", F.year(F.col("transaction_date")))
    .withColumn("transaction_month", F.month(F.col("transaction_date")))
    .withColumn("transaction_day", F.dayofmonth(F.col("transaction_date")))
    .withColumn("transaction_month_key", F.date_format(F.col("transaction_date"), "yyyy-MM"))
    .withColumn("days_since_transaction", F.datediff(analysis_date, F.col("transaction_date")))
    .withColumn("processing_timestamp", F.current_timestamp())
)

df_date_features.select(
    "transaction_id",
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "transaction_day",
    "transaction_month_key",
    "days_since_transaction",
    "processing_timestamp"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 9, Finished, Available, Finished, False)

+--------------+----------------+----------------+-----------------+---------------+---------------------+----------------------+--------------------------+
|transaction_id|transaction_date|transaction_year|transaction_month|transaction_day|transaction_month_key|days_since_transaction|processing_timestamp      |
+--------------+----------------+----------------+-----------------+---------------+---------------------+----------------------+--------------------------+
|T001          |2026-05-01      |2026            |5                |1              |2026-05              |9                     |2026-05-31 10:02:10.898279|
|T002          |2026-05-01      |2026            |5                |1              |2026-05              |9                     |2026-05-31 10:02:10.898279|
|T003          |2026-05-02      |2026            |5                |2              |2026-05              |8                     |2026-05-31 10:02:10.898279|
|T004          |2026-05-03      |2026            |5       

### Example 6: Build a standardized transaction DataFrame

ขั้นตอนนี้เลือก columns ที่ clean แล้ว และยังเก็บ raw fields สำคัญไว้บางส่วนเพื่อ debug ได้

ในงานจริง เราอาจเก็บ raw fields ไว้ใน Bronze และเลือกเฉพาะ standardized fields ใน Silver แต่ใน notebook วันนี้จะเก็บให้เห็นทั้งคู่เพื่อการเรียนรู้


In [8]:
df_standardized_transactions = (
    df_date_features
    .select(
        "transaction_id",
        "customer_id",
        "customer_key",
        "customer_name_clean",
        "email_clean",
        "phone_normalized",
        "status_clean",
        "customer_code_clean",
        "amount",
        "transaction_date_raw",
        "transaction_date",
        "date_parse_status",
        "event_timestamp_raw",
        "event_timestamp",
        "timestamp_parse_status",
        "transaction_year",
        "transaction_month",
        "transaction_month_key",
        "days_since_transaction",
        "source_system",
        "source_event",
        "source_region",
        "processing_timestamp"
    )
)

df_standardized_transactions.show(truncate=False)
df_standardized_transactions.printSchema()
print("Standardized row count:", df_standardized_transactions.count())

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 10, Finished, Available, Finished, False)

+--------------+-----------+------------+-------------------+-----------------------+----------------+------------+-------------------+------+--------------------+----------------+-----------------+-------------------+-------------------+----------------------+----------------+-----------------+---------------------+----------------------+-------------+------------+-------------+--------------------------+
|transaction_id|customer_id|customer_key|customer_name_clean|email_clean            |phone_normalized|status_clean|customer_code_clean|amount|transaction_date_raw|transaction_date|date_parse_status|event_timestamp_raw|event_timestamp    |timestamp_parse_status|transaction_year|transaction_month|transaction_month_key|days_since_transaction|source_system|source_event|source_region|processing_timestamp      |
+--------------+-----------+------------+-------------------+-----------------------+----------------+------------+-------------------+------+--------------------+----------------+

### Example 7: Check parse quality before writing

ก่อน write table ควรตรวจว่า parse issue มีจำนวนเท่าไร

การ check แบบนี้ช่วยให้เราไม่เผลอเอา invalid date/timestamp ไปใช้ต่อโดยไม่รู้ตัว


In [9]:
df_parse_quality_summary = (
    df_standardized_transactions
    .groupBy("date_parse_status", "timestamp_parse_status")
    .agg(F.count("*").alias("record_count"))
    .orderBy("date_parse_status", "timestamp_parse_status")
)

df_parse_quality_summary.show(truncate=False)

df_parse_issues = (
    df_standardized_transactions
    .filter(
        (F.col("date_parse_status") != "parsed") |
        (F.col("timestamp_parse_status") != "parsed")
    )
)

df_parse_issues.select(
    "transaction_id",
    "transaction_date_raw",
    "date_parse_status",
    "event_timestamp_raw",
    "timestamp_parse_status"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 11, Finished, Available, Finished, False)

+-----------------+----------------------+------------+
|date_parse_status|timestamp_parse_status|record_count|
+-----------------+----------------------+------------+
|missing          |missing               |1           |
|parsed           |parsed                |5           |
|unparsed         |unparsed              |1           |
+-----------------+----------------------+------------+

+--------------+--------------------+-----------------+-------------------+----------------------+
|transaction_id|transaction_date_raw|date_parse_status|event_timestamp_raw|timestamp_parse_status|
+--------------+--------------------+-----------------+-------------------+----------------------+
|T005          |not_available       |unparsed         |not_available      |unparsed              |
|T007          |                    |missing          |                   |missing               |
+--------------+--------------------+-----------------+-------------------+----------------------+



### Example 8: Write standardized result to a Lakehouse table

`saveAsTable()` จะเขียน output เป็น Lakehouse table ถ้า notebook attach กับ Fabric Lakehouse แล้ว

ถ้ายังไม่ได้ attach Lakehouse หรือ user ไม่มี permission cell นี้อาจ fail ได้ ให้ attach Lakehouse ก่อน run


In [10]:
table_name = "day08_standardized_transactions"

(
    df_standardized_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

print(f"Table written: {table_name}")

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 12, Finished, Available, Finished, False)

Table written: day08_standardized_transactions


### Example 9: Read table back from Lakehouse

หลัง write แล้ว ควรลอง read กลับเพื่อตรวจว่า table ใช้งานได้จริง


In [13]:
df_day08_table = spark.read.table("day08_standardized_transactions")

df_day08_table.select(
    "transaction_id",
    "customer_key",
    "status_clean",
    "transaction_date",
    "transaction_month_key",
    "date_parse_status",
    "timestamp_parse_status"
).show(truncate=False)

print("Lakehouse table row count:", df_day08_table.count())

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 15, Finished, Available, Finished, False)

+--------------+------------+------------+----------------+---------------------+-----------------+----------------------+
|transaction_id|customer_key|status_clean|transaction_date|transaction_month_key|date_parse_status|timestamp_parse_status|
+--------------+------------+------------+----------------+---------------------+-----------------+----------------------+
|T003          |CUST-0103   |success     |2026-05-02      |2026-05              |parsed           |parsed                |
|T006          |CUST-0106   |success     |2026-05-04      |2026-05              |parsed           |parsed                |
|T001          |CUST-0101   |success     |2026-05-01      |2026-05              |parsed           |parsed                |
|T002          |CUST-0102   |failed      |2026-05-01      |2026-05              |parsed           |parsed                |
|T004          |CUST-0104   |pending     |2026-05-03      |2026-05              |parsed           |parsed                |
|T005          |

## Quick recap

| Question | Answer |
|---|---|
| Date string ควรใช้ function อะไร parse? | `to_date()` พร้อมระบุ format |
| Timestamp string ควรใช้ function อะไร parse? | `to_timestamp()` พร้อมระบุ format |
| ถ้า source มีหลาย date format ควรทำอย่างไร? | ใช้ `coalesce()` ลอง parse หลาย pattern และสร้าง parse status |
| ทำไมต้อง `trim` / `lower` / `upper`? | เพื่อให้ string fields filter/join/group ได้สม่ำเสมอ |
| ใช้ `regexp_replace` ทำอะไรได้บ่อย? | ลบ special characters เช่น format phone/account/code |
| `current_timestamp()` คือเวลาอะไร? | Processing time ของ Spark job ไม่ใช่ event time จาก source |


## Exercises


### Exercise 1: Clean customer contact fields

สร้าง DataFrame ชื่อ `df_ex1_contact_cleaned` จาก `df_transactions_raw`

Requirements:

- สร้าง `customer_name_clean` ด้วย `trim` และ `initcap`
- สร้าง `email_clean` ด้วย `trim` และ `lower`
- สร้าง `status_clean` ด้วย `trim` และ `lower`
- สร้าง `phone_digits` โดยเก็บเฉพาะตัวเลข
- แสดงผลด้วย `.show(truncate=False)`

Expected result:

- ไม่มีช่องว่างหัวท้ายในชื่อ/email/status
- email และ status เป็น lowercase
- phone_digits เหลือเฉพาะตัวเลข


In [14]:
df_ex1_contact_cleaned = (
    df_transactions_raw
    .withColumn("customer_name_clean", F.initcap(F.trim(F.col("customer_name_raw"))))
    .withColumn("email_clean", F.lower(F.trim(F.col("email_raw"))))
    .withColumn("status_clean", F.lower(F.trim(F.col("status_raw"))))
    .withColumn("phone_digits", F.regexp_replace(F.col("phone_raw"), "[^0-9]", ""))
)

df_ex1_contact_cleaned.select(
    "transaction_id",
    "customer_name_raw",
    "customer_name_clean",
    "email_raw",
    "email_clean",
    "status_raw",
    "status_clean",
    "phone_raw",
    "phone_digits"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 16, Finished, Available, Finished, False)

+--------------+-----------------+-------------------+-----------------------+-----------------------+----------+------------+----------------+------------+
|transaction_id|customer_name_raw|customer_name_clean|email_raw              |email_clean            |status_raw|status_clean|phone_raw       |phone_digits|
+--------------+-----------------+-------------------+-----------------------+-----------------------+----------+------------+----------------+------------+
|T001          |  Alice Wong     |Alice Wong         |ALICE.WONG@EXAMPLE.COM |alice.wong@example.com | success  |success     |+66 81-234-5678 |66812345678 |
|T002          |bob chan         |Bob Chan           | Bob.Chan@example.com  |bob.chan@example.com   |FAILED    |failed      |081 999 0000    |0819990000  |
|T003          | CHARLIE TAN     |Charlie Tan        |CHARLIE.TAN@EXAMPLE.COM|charlie.tan@example.com|Success   |success     |(+66)82 555 8899|66825558899 |
|T004          |dara lee         |Dara Lee           | dar

### Exercise 2: Parse date and timestamp safely

สร้าง DataFrame ชื่อ `df_ex2_datetime_parsed` จาก `df_transactions_raw`

Requirements:

- parse `transaction_date_raw` เป็น `transaction_date`
- parse `event_timestamp_raw` เป็น `event_timestamp`
- รองรับอย่างน้อย 3 date/timestamp patterns ที่มีใน mock data
- สร้าง `date_parse_status` และ `timestamp_parse_status`
- แสดงเฉพาะ records ที่ status ไม่ใช่ `parsed`

Expected result:

- records ที่ parse ไม่ได้หรือ missing ต้องถูกมองเห็น
- ไม่ควร drop records ทิ้งเงียบ ๆ


In [15]:
df_ex2_datetime_parsed = (
    df_transactions_raw
    .withColumn(
        "transaction_date",
        F.coalesce(
            F.to_date(F.col("transaction_date_raw"), "yyyy-MM-dd"),
            F.to_date(F.col("transaction_date_raw"), "dd/MM/yyyy"),
            F.to_date(F.col("transaction_date_raw"), "yyyy/MM/dd")
        )
    )
    .withColumn(
        "event_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "dd/MM/yyyy HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy/MM/dd HH:mm:ss"),
            F.to_timestamp(F.col("event_timestamp_raw"), "yyyy-MM-dd'T'HH:mm:ss")
        )
    )
    .withColumn(
        "date_parse_status",
        F.when(
            (F.col("transaction_date").isNull()) & (F.length(F.trim(F.col("transaction_date_raw"))) > 0),
            F.lit("unparsed")
        ).when(
            F.length(F.trim(F.col("transaction_date_raw"))) == 0,
            F.lit("missing")
        ).otherwise(F.lit("parsed"))
    )
    .withColumn(
        "timestamp_parse_status",
        F.when(
            (F.col("event_timestamp").isNull()) & (F.length(F.trim(F.col("event_timestamp_raw"))) > 0),
            F.lit("unparsed")
        ).when(
            F.length(F.trim(F.col("event_timestamp_raw"))) == 0,
            F.lit("missing")
        ).otherwise(F.lit("parsed"))
    )
)

df_ex2_datetime_parsed.filter(
    (F.col("date_parse_status") != "parsed") |
    (F.col("timestamp_parse_status") != "parsed")
).select(
    "transaction_id",
    "transaction_date_raw",
    "transaction_date",
    "date_parse_status",
    "event_timestamp_raw",
    "event_timestamp",
    "timestamp_parse_status"
).show(truncate=False)

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 17, Finished, Available, Finished, False)

+--------------+--------------------+----------------+-----------------+-------------------+---------------+----------------------+
|transaction_id|transaction_date_raw|transaction_date|date_parse_status|event_timestamp_raw|event_timestamp|timestamp_parse_status|
+--------------+--------------------+----------------+-----------------+-------------------+---------------+----------------------+
|T005          |not_available       |NULL            |unparsed         |not_available      |NULL           |unparsed              |
|T007          |                    |NULL            |missing          |                   |NULL           |missing               |
+--------------+--------------------+----------------+-----------------+-------------------+---------------+----------------------+



### Exercise 3: Create a curated transaction table

สร้าง DataFrame ชื่อ `df_ex3_curated_transactions` โดยรวม logic จาก exercise ก่อนหน้า และเขียนเป็น Lakehouse table ชื่อ `day08_exercise_curated_transactions`

Requirements:

- มี `customer_key`
- มี `email_clean`
- มี `status_clean`
- มี `transaction_date`
- มี `transaction_month_key`
- มี `processing_timestamp`
- write ด้วย `.saveAsTable()`
- read table กลับมาตรวจด้วย `.show()` และ `.count()`

Expected result:

- table ถูกสร้างใน Lakehouse ถ้า notebook attach Lakehouse แล้ว
- row count หลัง read กลับควรเท่ากับ raw row count


In [16]:
df_ex3_curated_transactions = (
    df_ex2_datetime_parsed
    .withColumn("customer_key", F.concat(F.lit("CUST-"), F.lpad(F.col("customer_id").cast("string"), 4, "0")))
    .withColumn("email_clean", F.lower(F.trim(F.col("email_raw"))))
    .withColumn("status_clean", F.lower(F.trim(F.col("status_raw"))))
    .withColumn("transaction_month_key", F.date_format(F.col("transaction_date"), "yyyy-MM"))
    .withColumn("processing_timestamp", F.current_timestamp())
    .select(
        "transaction_id",
        "customer_id",
        "customer_key",
        "email_clean",
        "status_clean",
        "amount",
        "transaction_date",
        "transaction_month_key",
        "event_timestamp",
        "date_parse_status",
        "timestamp_parse_status",
        "processing_timestamp"
    )
)

df_ex3_curated_transactions.write.format("delta").mode("overwrite").saveAsTable("day08_exercise_curated_transactions")

df_ex3_table = spark.read.table("day08_exercise_curated_transactions")
df_ex3_table.show(truncate=False)
print("Exercise table row count:", df_ex3_table.count())

StatementMeta(, 6addfd8b-abd2-4f2e-a6d2-9777ccbc6173, 18, Finished, Available, Finished, False)

+--------------+-----------+------------+-----------------------+------------+------+----------------+---------------------+-------------------+-----------------+----------------------+--------------------------+
|transaction_id|customer_id|customer_key|email_clean            |status_clean|amount|transaction_date|transaction_month_key|event_timestamp    |date_parse_status|timestamp_parse_status|processing_timestamp      |
+--------------+-----------+------------+-----------------------+------------+------+----------------+---------------------+-------------------+-----------------+----------------------+--------------------------+
|T003          |103        |CUST-0103   |charlie.tan@example.com|success     |3400.0|2026-05-02      |2026-05              |2026-05-02 14:20:00|parsed           |parsed                |2026-05-31 10:19:01.481635|
|T001          |101        |CUST-0101   |alice.wong@example.com |success     |1200.5|2026-05-01      |2026-05              |2026-05-01 09:15:30|pars

## Common mistakes

1. **ใช้ `cast("date")` กับ raw string หลาย format โดยไม่ตรวจผลลัพธ์**

   ถ้า source มีหลาย date format การ cast ตรง ๆ อาจ parse ได้บาง records และกลายเป็น `null` บาง records โดยไม่รู้ตัว

2. **ไม่สร้าง parse status หลัง `to_date` / `to_timestamp`**

   Date ที่ parse ไม่ได้คือ data quality signal ควร flag ให้เห็น ไม่ควรปล่อยให้หายไปใน transformation ต่อ ๆ ไป

3. **ใช้ `current_timestamp()` แทน event time**

   `current_timestamp()` คือ processing time ของ job ไม่ใช่เวลาที่ transaction เกิดขึ้นจริง ถ้าใช้ผิดจะทำให้ timeline และ reporting เพี้ยน

4. **ลืม `trim()` ก่อน `lower()` / `upper()`**

   ตัวพิมพ์ถูกแล้วแต่ยังมีช่องว่างหัวท้าย อาจทำให้ join/filter ไม่ match อยู่ดี

5. **Normalize string แล้ว drop raw column เร็วเกินไป**

   ช่วง cleansing/debug ควรเก็บ raw value ไว้ก่อน เพื่อ trace ได้ว่าค่า clean มาจากอะไร

6. **ไม่คิดเรื่อง timezone ตอนใช้ timestamp**

   - ถ้าข้อมูลมาจากหลาย timezone อย่า assume ว่าทุกค่าเป็นเวลาไทยเสมอ ควรกำหนดให้ชัดว่า timestamp ที่ใช้ใน pipeline จะอ้างอิง UTC, Asia/Bangkok หรือ source local time
   - ถ้า source ส่งเวลามาเป็น local time ควรมี `source_timezone` ประกอบ และถ้าเป็นไปได้ควร normalize เป็น UTC ก่อนนำไปคำนวณหรือทำ report


## Expected Output / Success Criteria

เมื่อจบ Day 08 ควรทำได้:

- อธิบายความต่างระหว่าง raw string, DateType, TimestampType, event time และ processing time ได้
- ใช้ `trim`, `lower`, `upper`, `regexp_replace`, `split`, `concat` เพื่อ standardize string fields ได้
- parse date หลาย format ด้วย `to_date` + `coalesce` ได้
- parse timestamp หลาย format ด้วย `to_timestamp` + `coalesce` ได้
- สร้าง parse status column เพื่อแยก records ที่ parse date/timestamp สำเร็จ ออกจาก records ที่ missing หรือ parse ไม่สำเร็จได้
- derive date helper columns เช่น `transaction_year`, `transaction_month`, `transaction_month_key`, `days_since_transaction` ได้
- เขียน standardized DataFrame เป็น Lakehouse table และ read กลับมาตรวจได้ ถ้า notebook attach Lakehouse แล้ว
- optional exercise table `day08_exercise_curated_transactions` มี row count เท่ากับ raw input


## Final takeaway

Date, timestamp และ string functions เป็นพื้นฐานที่ดูเล็ก แต่กระทบ pipeline จริงเยอะมาก

> ก่อนใช้ raw field ใน business logic ต้องถามเสมอว่า field นี้ถูก parse, standardized และ validated แล้วหรือยัง

Mindset ที่ควรจำ:

- Date/time ที่ยังเป็น string ยังไม่พร้อมใช้จริง
- String ที่ยังไม่ trim/normalize อาจทำให้ join/filter ผิดได้
- Parse failure ต้องมองเห็น ไม่ใช่ถูกซ่อนไว้เป็น `null` เฉย ๆ
- Event time กับ processing time ใช้คนละวัตถุประสงค์
- เก็บ raw value ไว้ให้ debug ได้ก่อนค่อยตัดทิ้งในขั้นที่มั่นใจ


## Optional cleanup


In [ ]:
# spark.sql("DROP TABLE IF EXISTS day08_standardized_transactions")
# spark.sql("DROP TABLE IF EXISTS day08_exercise_curated_transactions")